# NB35 — PAH Panel: Literatur-Destekli Feature Engineering + Ablasyon

**TEKNOFEST Saglikta Yapay Zeka | Genetik Varyant Patojenite Tahmini**

## Amac

NB21 P4_COMBINED_BalBag modeli (MCC=0.529, Boot-F1=0.582) FE olmadan calisiyor.
Bu notebook, PAH'a ozel 21 feature (5 grup, missingness haric) ekleyerek MCC'yi iyilestirmeyi hedefler.

### Feature Gruplari
- **Grup A** (2): Frekans ozellikleri (has_any_freq, log_max_freq)
- **Grup B** (5): EK skor birlesimleri (ek_mean_all, ek_mean_top3, ek_max, ek_consensus, ek_delta_12)
- **Grup D** (5): AA fizikokimyasal delta (hydro, vol, mw, disorder, accessibility)
- **Grup E** (5): AA substitusyon skorlari (grantham, blosum62, grantham_cat, charge_change, proline_involved)
- **Grup F** (4): Prediktor uyumsuzlugu (ek_delta_7_1, ek_range, ek_std, ek_7_x_vol)
- **Grup C** (Missingness): CIKARILDI

### Deneyler
- **Exp 1**: Grup ablasyonu (her grubun solo katkisi)
- **Exp 2**: FE versiyon karsilastirmasi (no-FE vs NB16 FE vs full PAH FE vs optimal subset)

**Birincil metrik**: LOO-CV MCC
**Ikincil metrik**: Bootstrap %80/20 pathogenic-F1 (N=50)
**Baseline**: NB21 P4_COMBINED_BalBag MCC=0.529, Boot-F1=0.582

In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, gc
from datetime import datetime
from copy import deepcopy
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay,
    average_precision_score
)
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, REPORTS_DIR
from src import columns_real as CR
import lightgbm as lgb
try:
    from imblearn.ensemble import BalancedBaggingClassifier
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn", "-q"])
    from imblearn.ensemble import BalancedBaggingClassifier

np.random.seed(SEED)

PANEL             = "PAH"
HIGH_MISSING_THR  = 0.50
FINAL_BENIGN_FRAC = 0.80
N_BOOT            = 50
BOOT_SEED         = SEED
PI_TEST           = 0.20
N_MULTISEED       = 5

DATA_DIR    = os.path.join(PROJECT_ROOT, "data", "real_data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v18_pah_fe")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SEED         : {SEED}")
print(f"RESULTS_DIR  : {RESULTS_DIR}")

PROJECT_ROOT : /Users/tefe/teknofest_model/teknofest_model
SEED         : 42
RESULTS_DIR  : /Users/tefe/teknofest_model/teknofest_model/results/v18_pah_fe


In [2]:
# Cell 2: Veri Yukleme + Sutun Temizligi
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

def load_panel(name):
    return pd.read_csv(os.path.join(DATA_DIR, CR.PANEL_INFO[name]["file"]))

master   = load_panel("MASTER")
kanser   = load_panel("KANSER")
cftr_raw = load_panel("CFTR")
pah_raw  = load_panel("PAH")

feature_cols_all = [c for c in master.columns if c not in (ID_COL, TARGET)]

def drop_exact_duplicates(df, ref_df, name):
    ref_index = {}
    for _, row in ref_df.iterrows():
        key = (row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                                     for v in row[feature_cols_all].values)
        ref_index[key] = row[TARGET]
    drop_idx = []
    for idx, row in df.iterrows():
        key = (row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                                     for v in row[feature_cols_all].values)
        if key in ref_index and ref_index[key] == row[TARGET]:
            drop_idx.append(idx)
    df_clean = df.drop(index=drop_idx).reset_index(drop=True)
    print(f"{name}: {df.shape[0]} -> {df_clean.shape[0]} (drop={len(drop_idx)} birebir-ayni)")
    return df_clean

# PAH vs MASTER exact duplicates (expect 3 drops -> 369)
pah = drop_exact_duplicates(pah_raw, master, "PAH vs MASTER")

# KANSER vs PAH and CFTR vs PAH before combining
kanser_clean = drop_exact_duplicates(kanser, pah_raw, "KANSER vs PAH")
cftr_clean   = drop_exact_duplicates(cftr_raw, pah_raw, "CFTR vs PAH")

# COMBINED = MASTER + KANSER + CFTR (PAH excluded!)
combined = pd.concat([master, kanser_clean, cftr_clean], ignore_index=True)

print(f"\nMASTER  : {master.shape}, label: {master[TARGET].value_counts().to_dict()}")
print(f"KANSER  : {kanser_clean.shape}, label: {kanser_clean[TARGET].value_counts().to_dict()}")
print(f"CFTR    : {cftr_clean.shape}, label: {cftr_clean[TARGET].value_counts().to_dict()}")
print(f"PAH     : {pah.shape}, label: {pah[TARGET].value_counts().to_dict()}")
print(f"COMBINED: {combined.shape}, label: {combined[TARGET].value_counts().to_dict()}")

# Column cleanup from MASTER
constant_cols     = CR.get_constant_cols(master[feature_cols_all])
dup_pairs         = CR.get_duplicate_col_pairs(master[feature_cols_all])
dup_drop          = sorted({b for (a, b) in dup_pairs})
drop_cols         = sorted(set(constant_cols) | set(dup_drop))
base_feature_cols = [c for c in feature_cols_all if c not in drop_cols]
CAT_LIKE          = [c for c in (CR.CAT_COLS + CR.AA_COLS) if c in base_feature_cols]
NUM_COLS_BASE     = [c for c in base_feature_cols if c not in CAT_LIKE]

print(f"\nSutun temizligi: drop {len(drop_cols)} -> {len(base_feature_cols)} feature")
print(f"  ({len(NUM_COLS_BASE)} sayisal + {len(CAT_LIKE)} kategorik)")

PAH vs MASTER: 372 -> 369 (drop=3 birebir-ayni)
KANSER vs PAH: 388 -> 388 (drop=0 birebir-ayni)
CFTR vs PAH: 111 -> 111 (drop=0 birebir-ayni)

MASTER  : (2931, 353), label: {1: 2149, 0: 782}
KANSER  : (388, 353), label: {1: 268, 0: 120}
CFTR    : (111, 353), label: {1: 90, 0: 21}
PAH     : (369, 353), label: {1: 307, 0: 62}
COMBINED: (3430, 353), label: {1: 2507, 0: 923}

Sutun temizligi: drop 63 -> 288 feature
  (281 sayisal + 7 kategorik)


In [3]:
# Cell 3: Feature Engineering -- add_fe_pah() (21 feature, 5 grup)
AA_UNK      = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# === Grantham Distance Matrix ===
_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}
def grantham(a, b):
    if a == b: return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a)) or -1

# === BLOSUM62 Matrix ===
_B62_RAW = """A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4"""
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for _ri, _line in enumerate(_B62_RAW.strip().split("\n")):
    _toks = _line.split()
    _row_aa = _toks[0][0]
    _vals = [_toks[0][1:]] + _toks[1:]
    for _ci, _tok in enumerate(_vals):
        _col_aa = _ORDER[_ri + _ci]
        _v = int(_tok[1:] if _tok[0].isalpha() else _tok)
        _B62[(_row_aa, _col_aa)] = _v; _B62[(_col_aa, _row_aa)] = _v
def blosum62(a, b): return _B62.get((a, b), 0)

# === Physicochemical Lookup Tables ===
_HYDRO = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,
           'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,
           'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
_VOLUME = {'A':88.6,'R':173.4,'N':114.1,'D':111.1,'C':108.5,'Q':143.8,'E':138.4,'G':60.1,
            'H':153.2,'I':166.7,'L':166.7,'K':168.6,'M':162.9,'F':189.9,'P':112.7,'S':89.0,
            'T':116.1,'W':227.8,'Y':193.6,'V':140.0}
_MW = {'A':89.09,'R':174.20,'N':132.12,'D':133.10,'C':121.16,'Q':146.15,'E':147.13,'G':75.03,
       'H':155.16,'I':131.17,'L':131.17,'K':146.19,'M':149.21,'F':165.19,'P':115.13,'S':105.09,
       'T':119.12,'W':204.23,'Y':181.19,'V':117.15}
_CHARGE = {'R':'+','K':'+','H':'+','D':'-','E':'-',
           'A':'0','N':'0','C':'0','Q':'0','G':'0','I':'0','L':'0',
           'M':'0','F':'0','P':'0','S':'0','T':'0','W':'0','Y':'0','V':'0'}

# NEW for PAH: TOP-IDP disorder propensity
_DISORDER = {'A':0.06,'R':-0.18,'N':0.01,'D':0.05,'C':-0.20,'E':0.04,'Q':0.04,'G':0.17,
             'H':-0.08,'I':-0.49,'K':0.04,'L':-0.34,'M':-0.23,'F':-0.41,'P':0.41,'S':0.14,
             'T':-0.04,'V':-0.39,'W':-0.44,'Y':-0.27}

# NEW for PAH: Janin solvent accessibility
_ACCESSIBILITY = {'A':0.74,'R':0.64,'N':0.63,'D':0.62,'C':0.91,'E':0.62,'Q':0.62,'G':0.72,
                  'H':0.78,'I':0.88,'K':0.52,'L':0.85,'M':0.85,'F':0.88,'P':0.64,'S':0.66,
                  'T':0.70,'V':0.86,'W':0.85,'Y':0.76}

def detect_freq_cols(train_df):
    al_cols = [c for c in train_df.columns if c.startswith("AL_")]
    freq_cols = []
    for c in al_cols:
        vals = train_df[c].dropna()
        if len(vals) == 0: continue
        uniq = set(vals.unique())
        if len(uniq) > 2 and vals.min() >= 0 and vals.mean() < 0.1:
            freq_cols.append(c)
    print(f"Detected: {len(freq_cols)} freq cols from training data")
    return freq_cols

def add_fe_pah(df, freq_cols=None):
    out = df.copy()

    # === GROUP E: AA substitution scores (5 features) ===
    out["fe_grantham"] = out.apply(
        lambda r: grantham(r["AA_1"], r["AA_2"])
        if (isinstance(r["AA_1"], str) and isinstance(r["AA_2"], str)
            and r["AA_1"] in STANDARD_AA and r["AA_2"] in STANDARD_AA) else -1,
        axis=1).astype(float)
    out["fe_blosum62"] = out.apply(
        lambda r: blosum62(r["AA_1"], r["AA_2"])
        if (isinstance(r["AA_1"], str) and isinstance(r["AA_2"], str)
            and r["AA_1"] in STANDARD_AA and r["AA_2"] in STANDARD_AA) else 0,
        axis=1).astype(float)

    def _grantham_cat(val):
        if val < 0 or np.isnan(val): return np.nan
        if val == 0: return 0
        if val <= 60: return 1
        if val <= 80: return 2
        if val <= 100: return 3
        return 4
    out["fe_grantham_cat"] = out["fe_grantham"].apply(_grantham_cat)

    def _charge_change(row):
        x, y = row["AA_1"], row["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in _CHARGE and y in _CHARGE:
            cx, cy = _CHARGE[x], _CHARGE[y]
            if cx == cy: return 0
            if cx in ('+', '-') or cy in ('+', '-'): return 1
            return 0
        return np.nan
    out["fe_charge_change"] = out.apply(_charge_change, axis=1)

    # NEW: proline_involved (BMPR2-specific)
    def _proline_involved(row):
        x, y = row["AA_1"], row["AA_2"]
        if isinstance(x, str) and isinstance(y, str):
            return int(x == 'P' or y == 'P')
        return np.nan
    out["fe_proline_involved"] = out.apply(_proline_involved, axis=1)

    # === GROUP A: Frequency features (2 features) ===
    if freq_cols and len(freq_cols) > 0:
        max_pop_freq = out[freq_cols].max(axis=1)
        out["fe_has_any_freq"] = max_pop_freq.apply(lambda v: int(v > 0) if pd.notna(v) else 0)
        log_max_freq = np.log10(max_pop_freq.clip(lower=1e-10))
        log_max_freq = log_max_freq.where(max_pop_freq.notna(), np.nan)
        out["fe_log_max_freq"] = log_max_freq
    else:
        out["fe_has_any_freq"] = 0
        out["fe_log_max_freq"] = np.nan

    # === GROUP B: EK Score Combinations (5 features) ===
    ek_map = {}
    for i in range(1, 10):
        cn = f"EK_{i}"
        if cn in out.columns:
            ek_map[i] = out[cn]

    # fe_ek_mean_all: nanmean of all 9 EK columns
    ek_all_df = pd.DataFrame({f"EK_{i}": ek_map[i] for i in ek_map})
    out["fe_ek_mean_all"] = ek_all_df.mean(axis=1, skipna=True)

    if all(k in ek_map for k in [7, 9, 2]):
        out["fe_ek_mean_top3"] = pd.concat([ek_map[7], ek_map[9], ek_map[2]], axis=1).mean(axis=1, skipna=True)
    else:
        out["fe_ek_mean_top3"] = np.nan

    if all(k in ek_map for k in [7, 9, 2, 6]):
        four = pd.concat([ek_map[7], ek_map[9], ek_map[2], ek_map[6]], axis=1)
        out["fe_ek_max"] = four.max(axis=1, skipna=True)
    else:
        out["fe_ek_max"] = np.nan

    if all(k in ek_map for k in [4, 5, 6]):
        out["fe_ek_consensus"] = ek_map[4] + ek_map[5] + ek_map[6]
    else:
        out["fe_ek_consensus"] = np.nan

    if all(k in ek_map for k in [1, 2]):
        out["fe_ek_delta_12"] = ek_map[1] - ek_map[2]
    else:
        out["fe_ek_delta_12"] = np.nan

    # === GROUP D: AA Physicochemical Deltas (5 features) ===
    def _aa_delta_abs(lookup):
        def _calc(row):
            x, y = row["AA_1"], row["AA_2"]
            if isinstance(x, str) and isinstance(y, str) and x in lookup and y in lookup:
                return abs(lookup[x] - lookup[y])
            return np.nan
        return out.apply(_calc, axis=1)

    out["fe_hydro_abs"] = _aa_delta_abs(_HYDRO)
    out["fe_vol_abs"]   = _aa_delta_abs(_VOLUME)
    out["fe_mw_abs"]    = _aa_delta_abs(_MW)
    out["fe_disorder_abs"]     = _aa_delta_abs(_DISORDER)
    out["fe_accessibility_abs"] = _aa_delta_abs(_ACCESSIBILITY)

    # === GROUP F: Predictor Disagreement (4 features) — NEW for PAH ===
    if all(k in ek_map for k in [7, 1]):
        out["fe_ek_delta_7_1"] = ek_map[7] - ek_map[1]
    else:
        out["fe_ek_delta_7_1"] = np.nan

    out["fe_ek_range"] = ek_all_df.max(axis=1, skipna=True) - ek_all_df.min(axis=1, skipna=True)
    out["fe_ek_std"]   = ek_all_df.std(axis=1, skipna=True)

    # fe_ek_7_x_vol = EK_7 * fe_vol_abs
    if 7 in ek_map:
        out["fe_ek_7_x_vol"] = ek_map[7] * out["fe_vol_abs"]
    else:
        out["fe_ek_7_x_vol"] = np.nan

    return out

# Feature group definitions
FE_GROUP_A = ["fe_has_any_freq", "fe_log_max_freq"]
FE_GROUP_B = ["fe_ek_mean_all", "fe_ek_mean_top3", "fe_ek_max", "fe_ek_consensus", "fe_ek_delta_12"]
FE_GROUP_D = ["fe_hydro_abs", "fe_vol_abs", "fe_mw_abs", "fe_disorder_abs", "fe_accessibility_abs"]
FE_GROUP_E = ["fe_grantham", "fe_blosum62", "fe_grantham_cat", "fe_charge_change", "fe_proline_involved"]
FE_GROUP_F = ["fe_ek_delta_7_1", "fe_ek_range", "fe_ek_std", "fe_ek_7_x_vol"]
FE_ALL_PAH = FE_GROUP_A + FE_GROUP_B + FE_GROUP_D + FE_GROUP_E + FE_GROUP_F

print(f"add_fe_pah hazir. Gruplar: A={len(FE_GROUP_A)}, B={len(FE_GROUP_B)}, "
      f"D={len(FE_GROUP_D)}, E={len(FE_GROUP_E)}, F={len(FE_GROUP_F)}, Toplam={len(FE_ALL_PAH)}")
print(f"Feature listesi: {FE_ALL_PAH}")

add_fe_pah hazir. Gruplar: A=2, B=5, D=5, E=5, F=4, Toplam=21
Feature listesi: ['fe_has_any_freq', 'fe_log_max_freq', 'fe_ek_mean_all', 'fe_ek_mean_top3', 'fe_ek_max', 'fe_ek_consensus', 'fe_ek_delta_12', 'fe_hydro_abs', 'fe_vol_abs', 'fe_mw_abs', 'fe_disorder_abs', 'fe_accessibility_abs', 'fe_grantham', 'fe_blosum62', 'fe_grantham_cat', 'fe_charge_change', 'fe_proline_involved', 'fe_ek_delta_7_1', 'fe_ek_range', 'fe_ek_std', 'fe_ek_7_x_vol']


In [4]:
# Cell 4: M3 Preprocessing + Degerlendirme Altyapisi

LGBM_FIXED = {
    'n_estimators': 300, 'num_leaves': 31, 'learning_rate': 0.05,
    'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
    'verbose': -1, 'random_state': SEED, 'n_jobs': -1, 'importance_type': 'gain',
}

def fit_preprocessor(train_df_raw, fe_cols=None, freq_cols_list=None):
    if fe_cols:
        tr = add_fe_pah(train_df_raw, freq_cols=freq_cols_list)
    else:
        tr = train_df_raw.copy()
    fe_num = [c for c in (fe_cols or []) if c in tr.columns]
    num_cols = NUM_COLS_BASE + fe_num
    miss = tr[base_feature_cols].isna().mean()
    flag_source = miss[miss > HIGH_MISSING_THR].index.tolist()
    median = {c: pd.to_numeric(tr[c], errors="coerce").median() for c in num_cols}
    return {"num_cols": num_cols, "cat_cols": CAT_LIKE,
            "flag_source": flag_source, "median": median,
            "fe_cols": fe_cols or [], "freq_cols_list": freq_cols_list}

def transform_X(df_raw, pp):
    if pp["fe_cols"]:
        df = add_fe_pah(df_raw, freq_cols=pp["freq_cols_list"])
    else:
        df = df_raw.copy()
    out = pd.DataFrame(index=df.index)
    for c in pp["num_cols"]:
        out[c] = pd.to_numeric(df[c], errors="coerce").fillna(pp["median"].get(c, 0)).astype(float).values
    for c in pp["cat_cols"]:
        fill = AA_UNK if c in CR.AA_COLS else "MISSING"
        out[c] = df[c].astype("object").where(~df[c].isna(), fill).astype(str).values
    for c in pp["flag_source"]:
        out[CR.get_missing_mask_col_name(c)] = df_raw[c].isna().astype(int).values
    return out

def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y); prob = np.asarray(prob)
    neg = np.where(y == 0)[0]; pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0: return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED); f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    return {"mean": float(np.mean(f1s)), "std": float(np.std(f1s)),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_robust(y, prob, n=50):
    rng = np.random.RandomState(BOOT_SEED)
    thresholds = []
    for _ in range(n):
        yb, pb = _resample_8020(y, prob, rng)
        best, best_thr = -1.0, 0.5
        for thr in np.arange(0.05, 0.95, 0.01):
            f = _f1_pos(yb, (pb >= thr).astype(int))
            if f > best: best, best_thr = f, thr
        thresholds.append(best_thr)
    return float(np.mean(thresholds))

def _le_encode(X_df, cat_cols):
    Xn = X_df.copy()
    for c in cat_cols:
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c].astype(str))
    return Xn.astype(float)

def _get_estimator_from_bb(est):
    """BalancedBagging estimators are imblearn Pipelines with steps ['sampler', 'classifier']."""
    if hasattr(est, 'named_steps') and 'classifier' in est.named_steps:
        return est.named_steps['classifier']
    return est

def _make_balbag(seed=SEED):
    base = lgb.LGBMClassifier(**LGBM_FIXED)
    return BalancedBaggingClassifier(
        estimator=base, n_estimators=20,
        sampling_strategy="not minority",
        random_state=seed, n_jobs=-1
    )

def run_pah_eval(exp_name, combined_df, pah_df, fe_cols=None, freq_cols_list=None, seed=SEED):
    pp = fit_preprocessor(combined_df, fe_cols=fe_cols, freq_cols_list=freq_cols_list)
    X_combined = transform_X(combined_df, pp)
    y_combined = combined_df[TARGET].values
    X_pah = transform_X(pah_df, pp)
    y_pah = pah_df[TARGET].values
    cat_cols = list(pp["cat_cols"])

    X_comb_le = _le_encode(X_combined, cat_cols)
    X_pah_le  = _le_encode(X_pah, cat_cols)
    pi_train = float(y_combined.mean())

    bb = _make_balbag(seed=seed)
    bb.fit(X_comb_le, y_combined)
    p_raw = bb.predict_proba(X_pah_le)[:, 1]
    p_adj = adjust_prior_shift(p_raw, pi_train)

    thr = select_threshold_robust(y_pah, p_adj)
    y_pred = (p_adj >= thr).astype(int)
    mcc = matthews_corrcoef(y_pah, y_pred)
    f1 = _f1_pos(y_pah, y_pred)
    prec = precision_score(y_pah, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_pah, y_pred, pos_label=1, zero_division=0)
    auc = roc_auc_score(y_pah, p_adj) if len(np.unique(y_pah)) > 1 else 0.0
    boot = bootstrap_8020(y_pah, p_adj, thr)
    cm = confusion_matrix(y_pah, y_pred, labels=[0, 1])

    fi_arr = np.mean([_get_estimator_from_bb(est).feature_importances_ for est in bb.estimators_], axis=0)
    fi = pd.Series(fi_arr, index=X_comb_le.columns)

    return {
        "experiment": exp_name, "loo_mcc": round(mcc, 4),
        "f1": round(f1, 4), "precision": round(prec, 4), "recall": round(rec, 4),
        "auc": round(auc, 4), "thr": round(thr, 4),
        "boot_mean": round(boot["mean"], 4), "boot_std": round(boot["std"], 4),
        "boot_lo": round(boot["lo"], 4), "boot_hi": round(boot["hi"], 4),
        "TN": int(cm[0,0]), "FP": int(cm[0,1]), "FN": int(cm[1,0]), "TP": int(cm[1,1]),
        "n_features": X_comb_le.shape[1],
        "fi": fi, "y_pred": y_pred, "p_adj": p_adj
    }

print("Preprocessing + Degerlendirme altyapisi hazir.")
print(f"  BalancedBaggingClassifier(n_estimators=20, base=LGBMClassifier)")

Preprocessing + Degerlendirme altyapisi hazir.
  BalancedBaggingClassifier(n_estimators=20, base=LGBMClassifier)


In [5]:
# Cell 5: Frekans sutunlarini COMBINED uzerinde tespit et (leakage-free)
freq_cols = detect_freq_cols(combined)
print(f"Frekans sutun sayisi: {len(freq_cols)}")
print(f"Ilk 10: {freq_cols[:10]}")

Detected: 119 freq cols from training data
Frekans sutun sayisi: 119
Ilk 10: ['AL_1', 'AL_2', 'AL_3', 'AL_4', 'AL_5', 'AL_6', 'AL_7', 'AL_8', 'AL_9', 'AL_10']


In [6]:
# Cell 6: Exp 1 -- Grup Ablasyonu
print("="*70)
print("[Exp 1] PAH Feature Group Ablation")
print("="*70)

ablation_configs = {
    "E1a_GroupA_Freq":       FE_GROUP_A,
    "E1b_GroupB_EKCombo":    FE_GROUP_B,
    "E1c_GroupD_AAphyschem": FE_GROUP_D,
    "E1d_GroupE_AAsubst":    FE_GROUP_E,
    "E1e_GroupF_Disagree":   FE_GROUP_F,
    "E1f_All_ABDEF":         FE_ALL_PAH,
    "E1g_NoFE":              None,
}

all_results = {}

for exp_name, fe_cols_exp in ablation_configs.items():
    print(f"\n--- {exp_name} ({len(fe_cols_exp) if fe_cols_exp else 0} FE features) ---")
    try:
        res = run_pah_eval(exp_name, combined, pah,
                           fe_cols=fe_cols_exp, freq_cols_list=freq_cols)
        all_results[exp_name] = res
        print(f"  MCC={res['loo_mcc']:.4f}  Boot-F1={res['boot_mean']:.4f} +/- {res['boot_std']:.3f}  "
              f"Prec={res['precision']:.3f}  TN={res['TN']}  FP={res['FP']}  FN={res['FN']}  TP={res['TP']}")
    except Exception as e:
        print(f"  HATA: {e}")
        import traceback; traceback.print_exc()
        all_results[exp_name] = None

# Summary table
print("\n" + "="*70)
print("EXP 1 OZET:")
print(f"{'Deney':<25} {'MCC':>8} {'Boot-F1':>8} {'Boot-std':>9} {'Prec':>6} {'TN':>4} {'FP':>4} {'FN':>4} {'TP':>4}")
print("-"*80)
baseline_mcc = 0.529
baseline_bootf1 = 0.582
for name, res in all_results.items():
    if res is None: continue
    delta = res["loo_mcc"] - baseline_mcc
    print(f"{name:<25} {res['loo_mcc']:>8.4f} {res['boot_mean']:>8.4f} {res['boot_std']:>9.4f} "
          f"{res['precision']:>6.3f} {res['TN']:>4} {res['FP']:>4} {res['FN']:>4} {res['TP']:>4}  "
          f"({'+'if delta>=0 else ''}{delta:.4f})")
print("="*70)
print(f"NB21 Baseline: MCC=0.529, Boot-F1=0.582")

[Exp 1] PAH Feature Group Ablation

--- E1a_GroupA_Freq (2 FE features) ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.3686  Boot-F1=0.5329 +/- 0.062  Prec=0.942  TN=49  FP=13  FN=95  TP=212

--- E1b_GroupB_EKCombo (5 FE features) ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.3508  Boot-F1=0.5086 +/- 0.064  Prec=0.938  TN=48  FP=14  FN=97  TP=210

--- E1c_GroupD_AAphyschem (5 FE features) ---


...

  MCC=0.3399  Boot-F1=0.5080 +/- 0.073  Prec=0.940  TN=49  FP=13  FN=105  TP=202

--- E1f_All_ABDEF (21 FE features) ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.3807  Boot-F1=0.5388 +/- 0.077  Prec=0.950  TN=51  FP=11  FN=99  TP=208

--- E1g_NoFE (0 FE features) ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.3627  Boot-F1=0.5284 +/- 0.062  Prec=0.942  TN=49  FP=13  FN=97  TP=210

EXP 1 OZET:
Deney                          MCC  Boot-F1  Boot-std   Prec   TN   FP   FN   TP
--------------------------------------------------------------------------------
E1a_GroupA_Freq             0.3686   0.5329    0.0624  0.942   49   13   95  212  (-0.1604)
E1b_GroupB_EKCombo          0.3508   0.5086    0.0637  0.938   48   14   97  210  (-0.1782)
E1c_GroupD_AAphyschem       0.3596   0.5187    0.0604  0.938   48   14   94  213  (-0.1694)
E1d_GroupE_AAsubst          0.3546   0.5261    0.0708  0.944   50   12  104  203  (-0.1744)
E1e_GroupF_Disagree         0.3399   0.5080    0.0730  0.940   49   13  105  202  (-0.1891)
E1f_All_ABDEF               0.3807   0.5388    0.0768  0.950   51   11   99  208  (-0.1483)
E1g_NoFE                    0.3627   0.5284    0.0618  0.942   49   13   97  210  (-0.1663)
NB21 Baseline: MCC=0.529, Boot-F1=0.582


In [7]:
# Cell 7: Exp 2 -- FE Versiyon Karsilastirmasi
print("\n" + "="*70)
print("[Exp 2] FE Versiyon Karsilastirmasi")
print("="*70)

FE_NB16_PAH = ["fe_grantham", "fe_blosum62"]

# En iyi 2 grup: Exp 1 sonuclarina gore (auto-select by MCC, excluding NoFE and All)
e1_scored = [(k, v["loo_mcc"]) for k, v in all_results.items()
             if v and k not in ("E1g_NoFE", "E1f_All_ABDEF")]
e1_scored.sort(key=lambda x: x[1], reverse=True)
group_map = {
    "E1a_GroupA_Freq": FE_GROUP_A,
    "E1b_GroupB_EKCombo": FE_GROUP_B,
    "E1c_GroupD_AAphyschem": FE_GROUP_D,
    "E1d_GroupE_AAsubst": FE_GROUP_E,
    "E1e_GroupF_Disagree": FE_GROUP_F,
}
if len(e1_scored) >= 2:
    best2_names = [e1_scored[0][0], e1_scored[1][0]]
    best2_cols = []
    for n in best2_names:
        best2_cols.extend(group_map.get(n, []))
    print(f"En iyi 2 grup: {best2_names} -> {len(best2_cols)} feature")
else:
    best2_cols = FE_ALL_PAH

exp2_configs = {
    "E2a_NoFE":          None,
    "E2b_NB16_FE":       FE_NB16_PAH,
    "E2c_Full_PAH_FE":   FE_ALL_PAH,
    "E2d_Best2_Groups":  best2_cols if best2_cols else FE_ALL_PAH,
    "E2e_F_plus_D":      FE_GROUP_F + FE_GROUP_D,
}

for exp_name, fe_cols_exp in exp2_configs.items():
    print(f"\n--- {exp_name} ({len(fe_cols_exp) if fe_cols_exp else 0} FE features) ---")
    try:
        res = run_pah_eval(exp_name, combined, pah,
                           fe_cols=fe_cols_exp, freq_cols_list=freq_cols)
        all_results[exp_name] = res
        print(f"  MCC={res['loo_mcc']:.4f}  Boot-F1={res['boot_mean']:.4f} +/- {res['boot_std']:.3f}  "
              f"Prec={res['precision']:.3f}  TN={res['TN']}  FP={res['FP']}  FN={res['FN']}  TP={res['TP']}")
    except Exception as e:
        print(f"  HATA: {e}")
        import traceback; traceback.print_exc()
        all_results[exp_name] = None

print("\n" + "="*70)
print("EXP 2 OZET:")
print(f"{'Deney':<25} {'MCC':>8} {'Boot-F1':>8} {'Prec':>6} {'FP':>4} {'FN':>4} {'n_feat':>7}")
print("-"*65)
for name, res in all_results.items():
    if res is None or not name.startswith("E2"): continue
    print(f"{name:<25} {res['loo_mcc']:>8.4f} {res['boot_mean']:>8.4f} "
          f"{res['precision']:>6.3f} {res['FP']:>4} {res['FN']:>4} {res['n_features']:>7}")
print("="*70)


[Exp 2] FE Versiyon Karsilastirmasi
En iyi 2 grup: ['E1a_GroupA_Freq', 'E1c_GroupD_AAphyschem'] -> 7 feature

--- E2a_NoFE (0 FE features) ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.3627  Boot-F1=0.5284 +/- 0.062  Prec=0.942  TN=49  FP=13  FN=97  TP=210

--- E2b_NB16_FE (2 FE features) ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.3631  Boot-F1=0.5328 +/- 0.068  Prec=0.945  TN=50  FP=12  FN=101  TP=206

--- E2c_Full_PAH_FE (21 FE features) ---


...

  MCC=0.3807  Boot-F1=0.5388 +/- 0.077  Prec=0.950  TN=51  FP=11  FN=99  TP=208

--- E2d_Best2_Groups (7 FE features) ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.3745  Boot-F1=0.5312 +/- 0.065  Prec=0.943  TN=49  FP=13  FN=93  TP=214

--- E2e_F_plus_D (9 FE features) ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.3518  Boot-F1=0.5164 +/- 0.071  Prec=0.944  TN=50  FP=12  FN=105  TP=202

EXP 2 OZET:
Deney                          MCC  Boot-F1   Prec   FP   FN  n_feat
-----------------------------------------------------------------
E2a_NoFE                    0.3627   0.5284  0.942   13   97     428
E2b_NB16_FE                 0.3631   0.5328  0.945   12  101     430
E2c_Full_PAH_FE             0.3807   0.5388  0.950   11   99     449
E2d_Best2_Groups            0.3745   0.5312  0.943   13   93     435
E2e_F_plus_D                0.3518   0.5164  0.944   12  105     437


In [8]:
# Cell 8: Multi-seed Dogrulama
print("\n" + "="*70)
print("[Multi-seed Validation] Top 3 from Exp1+Exp2")
print("="*70)

# Select top 3 experiments by MCC
all_scored = [(k, v["loo_mcc"]) for k, v in all_results.items() if v]
all_scored.sort(key=lambda x: x[1], reverse=True)
top3_names = [x[0] for x in all_scored[:3]]
print(f"Top 3: {top3_names}")

# Get FE configs for top 3
def _get_fe_cols(name):
    if name in ablation_configs:
        return ablation_configs[name]
    elif name in exp2_configs:
        return exp2_configs[name]
    return None

multiseed_results = {}
seeds = [SEED + i * 7 for i in range(N_MULTISEED)]

for exp_name in top3_names:
    fe = _get_fe_cols(exp_name)
    seed_scores = []
    for s in seeds:
        try:
            res = run_pah_eval(f"{exp_name}_s{s}", combined, pah,
                               fe_cols=fe, freq_cols_list=freq_cols, seed=s)
            seed_scores.append({
                "seed": s, "mcc": res["loo_mcc"],
                "boot_f1": res["boot_mean"], "boot_std": res["boot_std"]
            })
        except Exception as e:
            print(f"  HATA: {exp_name} seed={s}: {e}")
    multiseed_results[exp_name] = seed_scores
    mccs = [x["mcc"] for x in seed_scores]
    f1s  = [x["boot_f1"] for x in seed_scores]
    print(f"\n{exp_name}:")
    print(f"  MCC:     {np.mean(mccs):.4f} +/- {np.std(mccs):.4f}  (seeds: {[f'{m:.4f}' for m in mccs]})")
    print(f"  Boot-F1: {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}  (seeds: {[f'{f:.4f}' for f in f1s]})")

print("\n" + "="*70)
print("Multi-seed Ozet:")
print(f"{'Deney':<25} {'MCC_mean':>9} {'MCC_std':>8} {'F1_mean':>8} {'F1_std':>8}")
print("-"*65)
for name, scores in multiseed_results.items():
    mccs = [x["mcc"] for x in scores]
    f1s  = [x["boot_f1"] for x in scores]
    print(f"{name:<25} {np.mean(mccs):>9.4f} {np.std(mccs):>8.4f} {np.mean(f1s):>8.4f} {np.std(f1s):>8.4f}")
print("="*70)


[Multi-seed Validation] Top 3 from Exp1+Exp2
Top 3: ['E1f_All_ABDEF', 'E2c_Full_PAH_FE', 'E2d_Best2_Groups']


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/


E1f_All_ABDEF:
  MCC:     0.3878 +/- 0.0269  (seeds: ['0.3807', '0.3778', '0.3463', '0.4135', '0.4207'])
  Boot-F1: 0.5426 +/- 0.0179  (seeds: ['0.5388', '0.5469', '0.5104', '0.5546', '0.5625'])


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/


E2c_Full_PAH_FE:
  MCC:     0.3878 +/- 0.0269  (seeds: ['0.3807', '0.3778', '0.3463', '0.4135', '0.4207'])
  Boot-F1: 0.5426 +/- 0.0179  (seeds: ['0.5388', '0.5469', '0.5104', '0.5546', '0.5625'])


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/


E2d_Best2_Groups:
  MCC:     0.3753 +/- 0.0092  (seeds: ['0.3745', '0.3656', '0.3690', '0.3923', '0.3753'])
  Boot-F1: 0.5255 +/- 0.0135  (seeds: ['0.5312', '0.5193', '0.5144', '0.5494', '0.5133'])

Multi-seed Ozet:
Deney                      MCC_mean  MCC_std  F1_mean   F1_std
-----------------------------------------------------------------
E1f_All_ABDEF                0.3878   0.0269   0.5426   0.0179
E2c_Full_PAH_FE              0.3878   0.0269   0.5426   0.0179
E2d_Best2_Groups             0.3753   0.0092   0.5255   0.0135


In [9]:
# Cell 9: Feature Importance + Korelasyon Dogrulamasi
print("\n" + "="*70)
print("Feature Importance + Korelasyon Analizi")
print("="*70)

# En iyi FE modeli (exclude NoFE)
best_key = max(
    [(k, v["loo_mcc"]) for k, v in all_results.items() if v and k not in ("E1g_NoFE", "E2a_NoFE")],
    key=lambda x: x[1]
)[0]
best_res = all_results[best_key]
fi = best_res["fi"]

fe_fi = fi[[c for c in FE_ALL_PAH if c in fi.index]].sort_values(ascending=False)
print(f"\nEn iyi model: {best_key} (MCC={best_res['loo_mcc']:.4f})")
print(f"\nFE Feature Importance (top):")
for feat, imp in fe_fi.items():
    total_pct = imp / fi.sum() * 100
    print(f"  {feat:<25} {imp:>10.1f}  ({total_pct:>5.2f}%)")

print(f"\nTop-20 Overall Feature Importance:")
top20 = fi.sort_values(ascending=False).head(20)
for feat, imp in top20.items():
    marker = " <-- FE" if feat in FE_ALL_PAH else ""
    print(f"  {feat:<25} {imp:>10.1f}{marker}")

fi_df = pd.DataFrame({"feature": fi.index, "importance": fi.values})
fi_df = fi_df.sort_values("importance", ascending=False).reset_index(drop=True)
fi_df.to_csv(os.path.join(RESULTS_DIR, "feature_importance.csv"), index=False)

# Korelasyon dogrulamasi
print("\n" + "-"*70)
print("PAH FE Korelasyon Dogrulamasi")
pah_fe = add_fe_pah(pah, freq_cols=freq_cols)
y_pah_arr = pah[TARGET].values

print(f"\n{'Feature':<25} {'r(Label)':>10} {'n_valid':>8} {'Grup':<20}")
print("-"*70)
for feat in FE_ALL_PAH:
    if feat not in pah_fe.columns: continue
    vals = pd.to_numeric(pah_fe[feat], errors="coerce")
    mask = vals.notna()
    if mask.sum() < 10:
        print(f"  {feat:<25} {'N/A':>10} {mask.sum():>8}")
        continue
    r = np.corrcoef(vals[mask].values, y_pah_arr[mask])[0, 1]
    if feat in FE_GROUP_A: cat = "Grup A (Freq)"
    elif feat in FE_GROUP_B: cat = "Grup B (EK combo)"
    elif feat in FE_GROUP_D: cat = "Grup D (AA phys)"
    elif feat in FE_GROUP_E: cat = "Grup E (AA subst)"
    elif feat in FE_GROUP_F: cat = "Grup F (Disagree)"
    else: cat = "?"
    print(f"  {feat:<25} {r:>10.4f} {mask.sum():>8} {cat:<20}")


Feature Importance + Korelasyon Analizi

En iyi model: E1f_All_ABDEF (MCC=0.3807)

FE Feature Importance (top):
  fe_ek_7_x_vol                  842.3  ( 3.40%)
  fe_ek_delta_7_1                759.4  ( 3.06%)
  fe_ek_mean_top3                559.9  ( 2.26%)
  fe_grantham                    393.9  ( 1.59%)
  fe_ek_delta_12                 377.2  ( 1.52%)
  fe_ek_consensus                354.5  ( 1.43%)
  fe_vol_abs                     303.5  ( 1.22%)
  fe_ek_max                      291.7  ( 1.18%)
  fe_ek_mean_all                 229.8  ( 0.93%)
  fe_mw_abs                      218.4  ( 0.88%)
  fe_log_max_freq                215.1  ( 0.87%)
  fe_ek_std                      206.8  ( 0.83%)
  fe_blosum62                    202.0  ( 0.81%)
  fe_disorder_abs                196.1  ( 0.79%)
  fe_accessibility_abs           183.1  ( 0.74%)
  fe_ek_range                    173.3  ( 0.70%)
  fe_hydro_abs                   158.9  ( 0.64%)
  fe_charge_change                82.4  ( 0.33%)
  fe_

In [10]:
# Cell 10: Gorsellestirmeler + CSV

# Tum sonuclari topla
rows = []
for k, v in sorted(all_results.items()):
    if v is None: continue
    rows.append({k2: v2 for k2, v2 in v.items() if k2 not in ("fi", "y_pred", "p_adj")})
results_df = pd.DataFrame(rows)
csv_path = os.path.join(RESULTS_DIR, "pah_fe_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"Sonuclar: {csv_path}")
print(results_df[["experiment","loo_mcc","boot_mean","boot_std","precision","FP","FN","n_features"]].to_string(index=False))

# Save group ablation separately
e1_rows = [{k2: v2 for k2, v2 in v.items() if k2 not in ("fi", "y_pred", "p_adj")}
            for k, v in sorted(all_results.items()) if v and k.startswith("E1")]
if e1_rows:
    pd.DataFrame(e1_rows).to_csv(os.path.join(RESULTS_DIR, "group_ablation.csv"), index=False)
    print("group_ablation.csv kaydedildi.")

# Fig 1: Group Ablation (MCC + Boot-F1 side by side)
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig1.suptitle("NB35 - PAH Feature Engineering Ablation", fontweight="bold", fontsize=13)

e1_names = list(ablation_configs.keys())
e1_mcc = [all_results.get(k, {}).get("loo_mcc", 0) if all_results.get(k) else 0 for k in e1_names]
e1_boot = [all_results.get(k, {}).get("boot_mean", 0) if all_results.get(k) else 0 for k in e1_names]
colors = ['#2196F3' if k != "E1g_NoFE" else '#FF5722' for k in e1_names]
short_names = [n.split("_", 1)[1] for n in e1_names]

bars = ax1.bar(range(len(e1_names)), e1_mcc, color=colors, edgecolor="black", linewidth=0.8)
ax1.axhline(y=baseline_mcc, color='red', linestyle='--', linewidth=1.5, label=f'NB21 Baseline ({baseline_mcc})')
for bar, val in zip(bars, e1_mcc):
    if val > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)
ax1.set_xticks(range(len(e1_names)))
ax1.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel("MCC"); ax1.set_title("Exp 1: Grup Ablasyonu (MCC)")
ax1.set_ylim(0, 0.85); ax1.legend()

bars2 = ax2.bar(range(len(e1_names)), e1_boot, color=colors, edgecolor="black", linewidth=0.8)
ax2.axhline(y=baseline_bootf1, color='red', linestyle='--', linewidth=1.5, label=f'NB21 Baseline ({baseline_bootf1})')
for bar, val in zip(bars2, e1_boot):
    if val > 0:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)
ax2.set_xticks(range(len(e1_names)))
ax2.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel("Bootstrap 80/20 F1"); ax2.set_title("Exp 1: Grup Ablasyonu (Boot-F1)")
ax2.set_ylim(0, 1.05); ax2.legend()

plt.tight_layout()
fig1.savefig(os.path.join(RESULTS_DIR, "fig1_group_ablation.png"), dpi=150, bbox_inches="tight")
plt.close(fig1)
print("fig1_group_ablation.png kaydedildi.")

# Fig 2: FE Version Comparison
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 6))
fig2.suptitle("NB35 - PAH FE Version Comparison", fontweight="bold", fontsize=13)

e2_names = list(exp2_configs.keys())
e2_mcc = [all_results.get(k, {}).get("loo_mcc", 0) if all_results.get(k) else 0 for k in e2_names]
e2_boot = [all_results.get(k, {}).get("boot_mean", 0) if all_results.get(k) else 0 for k in e2_names]
e2_short = [n.split("_", 1)[1] for n in e2_names]
e2_colors = ['#FF5722', '#FFC107', '#4CAF50', '#2196F3', '#9C27B0']

bars3 = ax3.bar(range(len(e2_names)), e2_mcc, color=e2_colors[:len(e2_names)], edgecolor="black", linewidth=0.8)
ax3.axhline(y=baseline_mcc, color='red', linestyle='--', linewidth=1.5, label='NB21 Baseline')
for bar, val in zip(bars3, e2_mcc):
    if val > 0:
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
ax3.set_xticks(range(len(e2_names)))
ax3.set_xticklabels(e2_short, rotation=30, ha='right', fontsize=9)
ax3.set_ylabel("MCC"); ax3.set_title("Exp 2: MCC")
ax3.set_ylim(0, 0.85); ax3.legend()

bars4 = ax4.bar(range(len(e2_names)), e2_boot, color=e2_colors[:len(e2_names)], edgecolor="black", linewidth=0.8)
ax4.axhline(y=baseline_bootf1, color='red', linestyle='--', linewidth=1.5, label='NB21 Baseline')
for bar, val in zip(bars4, e2_boot):
    if val > 0:
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
ax4.set_xticks(range(len(e2_names)))
ax4.set_xticklabels(e2_short, rotation=30, ha='right', fontsize=9)
ax4.set_ylabel("Bootstrap 80/20 F1"); ax4.set_title("Exp 2: Boot-F1")
ax4.set_ylim(0, 1.05); ax4.legend()

plt.tight_layout()
fig2.savefig(os.path.join(RESULTS_DIR, "fig2_fe_comparison.png"), dpi=150, bbox_inches="tight")
plt.close(fig2)
print("fig2_fe_comparison.png kaydedildi.")

# Fig 3: Feature Importance (FE only, color-coded by group)
fe_fi = fi[[c for c in FE_ALL_PAH if c in fi.index]].sort_values(ascending=True)
if len(fe_fi) > 0:
    fig3, ax5 = plt.subplots(figsize=(8, max(4, len(fe_fi)*0.35)))
    colors_fi = []
    for feat in fe_fi.index:
        if feat in FE_GROUP_A: colors_fi.append('#2196F3')
        elif feat in FE_GROUP_B: colors_fi.append('#4CAF50')
        elif feat in FE_GROUP_D: colors_fi.append('#FF9800')
        elif feat in FE_GROUP_E: colors_fi.append('#9C27B0')
        elif feat in FE_GROUP_F: colors_fi.append('#E91E63')
        else: colors_fi.append('#607D8B')
    ax5.barh(range(len(fe_fi)), fe_fi.values, color=colors_fi, edgecolor="black", linewidth=0.5)
    ax5.set_yticks(range(len(fe_fi)))
    ax5.set_yticklabels(fe_fi.index, fontsize=8)
    ax5.set_xlabel("Feature Importance (gain)")
    ax5.set_title(f"NB35 - PAH FE Feature Importance ({best_key})")
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#2196F3', label='A: Freq'),
                      Patch(facecolor='#4CAF50', label='B: EK combo'),
                      Patch(facecolor='#FF9800', label='D: AA phys'),
                      Patch(facecolor='#9C27B0', label='E: AA subst'),
                      Patch(facecolor='#E91E63', label='F: Disagree')]
    ax5.legend(handles=legend_elements, loc='lower right', fontsize=8)
    plt.tight_layout()
    fig3.savefig(os.path.join(RESULTS_DIR, "fig3_feature_importance.png"), dpi=150, bbox_inches="tight")
    plt.close(fig3)
    print("fig3_feature_importance.png kaydedildi.")

# Fig 4: Confusion matrix (best model)
best_overall = max([(k, v) for k, v in all_results.items() if v], key=lambda x: x[1]["loo_mcc"])
best_name_o, best_r_o = best_overall
cm = np.array([[best_r_o["TN"], best_r_o["FP"]], [best_r_o["FN"], best_r_o["TP"]]])
fig4, ax6 = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=["Benign", "Pathogenic"])
disp.plot(ax=ax6, cmap="Blues", values_format="d")
ax6.set_title(f"Best: {best_name_o}\nMCC={best_r_o['loo_mcc']:.4f}, Boot-F1={best_r_o['boot_mean']:.4f}")
plt.tight_layout()
fig4.savefig(os.path.join(RESULTS_DIR, "fig4_confusion_best.png"), dpi=150, bbox_inches="tight")
plt.close(fig4)
print("fig4_confusion_best.png kaydedildi.")

# Fig 5: Multi-seed validation
if multiseed_results:
    fig5, (ax7, ax8) = plt.subplots(1, 2, figsize=(12, 5))
    fig5.suptitle("NB35 - PAH Multi-seed Validation", fontweight="bold", fontsize=13)
    ms_names = list(multiseed_results.keys())
    ms_short = [n.split("_", 1)[1] if "_" in n else n for n in ms_names]

    # MCC boxplot
    mcc_data = [[x["mcc"] for x in multiseed_results[n]] for n in ms_names]
    bp1 = ax7.boxplot(mcc_data, labels=ms_short, patch_artist=True)
    for patch in bp1['boxes']:
        patch.set_facecolor('#2196F3')
        patch.set_alpha(0.7)
    ax7.axhline(y=baseline_mcc, color='red', linestyle='--', linewidth=1.5, label='NB21 Baseline')
    ax7.set_ylabel("MCC"); ax7.set_title("Multi-seed MCC"); ax7.legend()
    ax7.tick_params(axis='x', rotation=30)

    # Boot-F1 boxplot
    f1_data = [[x["boot_f1"] for x in multiseed_results[n]] for n in ms_names]
    bp2 = ax8.boxplot(f1_data, labels=ms_short, patch_artist=True)
    for patch in bp2['boxes']:
        patch.set_facecolor('#4CAF50')
        patch.set_alpha(0.7)
    ax8.axhline(y=baseline_bootf1, color='red', linestyle='--', linewidth=1.5, label='NB21 Baseline')
    ax8.set_ylabel("Bootstrap 80/20 F1"); ax8.set_title("Multi-seed Boot-F1"); ax8.legend()
    ax8.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    fig5.savefig(os.path.join(RESULTS_DIR, "fig5_multiseed.png"), dpi=150, bbox_inches="tight")
    plt.close(fig5)
    print("fig5_multiseed.png kaydedildi.")

Sonuclar: /Users/tefe/teknofest_model/teknofest_model/results/v18_pah_fe/pah_fe_results.csv
           experiment  loo_mcc  boot_mean  boot_std  precision  FP  FN  n_features
      E1a_GroupA_Freq   0.3686     0.5329    0.0624     0.9422  13  95         430
   E1b_GroupB_EKCombo   0.3508     0.5086    0.0637     0.9375  14  97         433
E1c_GroupD_AAphyschem   0.3596     0.5187    0.0604     0.9383  14  94         433
   E1d_GroupE_AAsubst   0.3546     0.5261    0.0708     0.9442  12 104         433
  E1e_GroupF_Disagree   0.3399     0.5080    0.0730     0.9395  13 105         432
        E1f_All_ABDEF   0.3807     0.5388    0.0768     0.9498  11  99         449
             E1g_NoFE   0.3627     0.5284    0.0618     0.9417  13  97         428
             E2a_NoFE   0.3627     0.5284    0.0618     0.9417  13  97         428
          E2b_NB16_FE   0.3631     0.5328    0.0675     0.9450  12 101         430
      E2c_Full_PAH_FE   0.3807     0.5388    0.0768     0.9498  11  99        

In [11]:
# Cell 11: PDF Rapor
from fpdf import FPDF

class NB35Report(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 6, "NB35 - PAH Feature Engineering Ablation", align="C", new_x="LMARGIN", new_y="NEXT")
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)
    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", align="C")
    def section(self, title):
        self.set_font("Helvetica", "B", 12)
        self.cell(0, 8, title, new_x="LMARGIN", new_y="NEXT")
        self.ln(2)
    def body_text(self, txt):
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, txt)
        self.ln(2)
    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [190 // len(headers)] * len(headers)
        self.set_font("Helvetica", "B", 8)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), border=1, align="C")
        self.ln()
        self.set_font("Helvetica", "", 7)
        for row in rows:
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), border=1, align="C")
            self.ln()
        self.ln(3)
    def add_fig(self, path, w=170):
        if os.path.exists(path):
            self.image(path, w=w)
            self.ln(3)

pdf = NB35Report()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=20)

# === Page 1: Title ===
pdf.add_page()
pdf.set_font("Helvetica", "B", 16)
pdf.cell(0, 15, "NB35: PAH Panel", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "B", 13)
pdf.cell(0, 10, "Literatur-Destekli Feature Engineering", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.cell(0, 10, "Ablasyon Raporu", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 8, f"Tarih: {datetime.now().strftime('%Y-%m-%d %H:%M')}", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.ln(5)
pdf.body_text(
    "Baseline: NB21 P4_COMBINED_BalBag\n"
    "  MCC=0.529, Boot-F1=0.582\n"
    "  TN=46, FP=16, FN=45, TP=262\n"
    "  Training: COMBINED(3430) + BalancedBagging(20xLGBM)\n"
    "  Panel: PAH (372 rows -> 369 after dedup, 310 patho / 62 benign)"
)

# === Section 1: Motivasyon ===
pdf.add_page()
pdf.section("1. Motivasyon ve Arka Plan")
pdf.body_text(
    "PAH paneli projenin en zorlu parcasidir. Temel nedenler: n=62 benign (azinlik-ornegi tavani), "
    "zayif EK korelasyonlari (max |r|=0.284), MASTER ile gen-profil farki. "
    "Feature engineering, model degistirmeden sinyal/gurultu oranini artiran yontemdir. "
    "NB34 CFTR'de benzer yaklasim test edilmisti. PAH'ta ayni literatur-destekli FE kullanilir."
)

# === Section 2: Feature Details ===
pdf.section("2. Feature Gruplari Detayi (21 feature, 5 grup)")
pdf.body_text(
    "Grup A (2): Frekans - populasyon frekans bilgisi, ACMG BA1/BS1 kriterleri\n"
    "Grup B (5): EK Skor Birlesimleri - meta-prediktor kombinasyonlari\n"
    "Grup D (5): AA Fizikokimyasal Delta - disorder/accessibility yeni eklendi\n"
    "Grup E (5): AA Substitusyon Skorlari - Grantham, BLOSUM62, proline\n"
    "Grup F (4): Prediktor Uyumsuzlugu - PAH-spesifik, yeni grup\n"
    "Kaynak: Grantham 1974, MutPred2, PON-P3, BMPR2 structural studies"
)

# === Section 3: Exp 1 Results ===
pdf.add_page()
pdf.section("3. Exp 1 - Grup Ablasyonu")
headers_e1 = ["Deney", "MCC", "Boot-F1", "Prec", "FP", "FN"]
rows_e1 = []
for k in ablation_configs.keys():
    v = all_results.get(k)
    if v is None: continue
    rows_e1.append([
        v["experiment"], f"{v['loo_mcc']:.4f}", f"{v['boot_mean']:.4f}",
        f"{v['precision']:.3f}", str(v["FP"]), str(v["FN"])
    ])
pdf.add_table(headers_e1, rows_e1)
pdf.add_fig(os.path.join(RESULTS_DIR, "fig1_group_ablation.png"))

# === Section 4: Exp 2 Results ===
pdf.add_page()
pdf.section("4. Exp 2 - FE Versiyon Karsilastirmasi")
headers_e2 = ["Deney", "MCC", "Boot-F1", "n_feat"]
rows_e2 = []
for k in exp2_configs.keys():
    v = all_results.get(k)
    if v is None: continue
    rows_e2.append([
        v["experiment"], f"{v['loo_mcc']:.4f}", f"{v['boot_mean']:.4f}",
        str(v["n_features"])
    ])
pdf.add_table(headers_e2, rows_e2)
pdf.add_fig(os.path.join(RESULTS_DIR, "fig2_fe_comparison.png"))

# === Section 5: Feature Importance ===
pdf.add_page()
pdf.section("5. Feature Importance Analizi")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig3_feature_importance.png"))

# === Section 6: Confusion Matrix ===
pdf.section("6. Confusion Matrix")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig4_confusion_best.png"))

# === Section 7: Multi-seed ===
if multiseed_results:
    pdf.add_page()
    pdf.section("7. Multi-seed Dogrulama")
    pdf.add_fig(os.path.join(RESULTS_DIR, "fig5_multiseed.png"))

# === Section 8: Sonuc ===
pdf.section("8. Sonuc ve Karar")
best_all = max([(k,v) for k,v in all_results.items() if v], key=lambda x: x[1]["loo_mcc"])
delta = best_all[1]["loo_mcc"] - 0.529
pdf.body_text(
    f"En iyi model: {best_all[0]}\n"
    f"  MCC: {best_all[1]['loo_mcc']:.4f} vs 0.529 (NB21), Delta={delta:+.4f}\n"
    f"  Boot-F1: {best_all[1]['boot_mean']:.4f} vs 0.582 (NB21)\n\n"
    f"Karar: {'FE anlamli iyilestirme sagladi' if delta > 0.02 else ('FE kucuk iyilestirme, guerultue bandinda' if delta > 0 else 'FE deger katmadi')}\n\n"
    f"PAH'in temel zorluklari FE ile tam cozulmeyebilir: n=62 benign tavani ve zayif EK korelasyonlari."
)

pdf_path = os.path.join(REPORTS_DIR, "NB35_pah_fe_report.pdf")
pdf.output(pdf_path)
print(f"PDF rapor kaydedildi: {pdf_path}")

PDF rapor kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/NB35_pah_fe_report.pdf


In [12]:
# Cell 12: Ozet
print("\n" + "="*70)
print("NB35 TAMAMLANDI")
print("="*70)

best_all = max([(k,v) for k,v in all_results.items() if v], key=lambda x: x[1]["loo_mcc"])
baseline_mcc = 0.529
baseline_bootf1 = 0.582
delta = best_all[1]["loo_mcc"] - baseline_mcc

print(f"\nBaseline (NB21): MCC=0.529, Boot-F1=0.582")
print(f"En iyi FE    : {best_all[0]}")
print(f"  MCC        : {best_all[1]['loo_mcc']:.4f} (delta={'+' if delta>=0 else ''}{delta:.4f})")
print(f"  Boot-F1    : {best_all[1]['boot_mean']:.4f} +/- {best_all[1]['boot_std']:.4f}")
print(f"  Precision  : {best_all[1]['precision']:.4f}")
print(f"  CM         : TN={best_all[1]['TN']}, FP={best_all[1]['FP']}, FN={best_all[1]['FN']}, TP={best_all[1]['TP']}")

if delta > 0.02:
    print(f"\n>> KARAR: FE anlamli iyilestirme sagladi. En iyi FE sabitleniyor.")
elif delta > 0:
    print(f"\n>> KARAR: FE kucuk iyilestirme, guerultue bandinda. Ek dogrulama gerekir.")
else:
    print(f"\n>> KARAR: FE deger katmadi. NB21 baseline korunuyor.")

print(f"\nCiktilar: {RESULTS_DIR}/")
print(f"  pah_fe_results.csv")
print(f"  group_ablation.csv")
print(f"  feature_importance.csv")
print(f"  fig1_group_ablation.png")
print(f"  fig2_fe_comparison.png")
print(f"  fig3_feature_importance.png")
print(f"  fig4_confusion_best.png")
print(f"  fig5_multiseed.png")
print(f"  {REPORTS_DIR}/NB35_pah_fe_report.pdf")
print("="*70)


NB35 TAMAMLANDI

Baseline (NB21): MCC=0.529, Boot-F1=0.582
En iyi FE    : E1f_All_ABDEF
  MCC        : 0.3807 (delta=-0.1483)
  Boot-F1    : 0.5388 +/- 0.0768
  Precision  : 0.9498
  CM         : TN=51, FP=11, FN=99, TP=208

>> KARAR: FE deger katmadi. NB21 baseline korunuyor.

Ciktilar: /Users/tefe/teknofest_model/teknofest_model/results/v18_pah_fe/
  pah_fe_results.csv
  group_ablation.csv
  feature_importance.csv
  fig1_group_ablation.png
  fig2_fe_comparison.png
  fig3_feature_importance.png
  fig4_confusion_best.png
  fig5_multiseed.png
  /Users/tefe/teknofest_model/teknofest_model/reports/NB35_pah_fe_report.pdf
